# 02 - Statistics: do late deliveries get lower reviews?
**H0:** review scores of late and on-time orders come from the same distribution.
**H1:** they differ.

Review scores are ordinal (1-5) and heavily skewed, so I use the **Mann-Whitney U test** (no normality assumption) and report an
**effect size** and a **bootstrap confidence interval**, because with ~95k orders almost any difference is "significant".

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
fo = pd.read_csv(ROOT / "dashboard_data" / "fact_orders.csv")
d = fo[(fo.is_delivered == 1) & fo.is_late.notna() & fo.review_score.notna()].copy()
late   = d.loc[d.is_late == 1, "review_score"].to_numpy()
ontime = d.loc[d.is_late == 0, "review_score"].to_numpy()

print(pd.DataFrame({"n": [len(late), len(ontime)], "mean": [late.mean(), ontime.mean()],
                    "median": [np.median(late), np.median(ontime)]}, index=["Late", "On time"]).round(2))

             n  mean  median
Late      6381  2.27     1.0
On time  89443  4.29     5.0


In [2]:
u, p = stats.mannwhitneyu(late, ontime, alternative="two-sided")
rbc = 2 * u / (len(late) * len(ontime)) - 1          # rank-biserial correlation (negative = late scores lower)
print(f"Mann-Whitney U = {u:,.0f}, p-value = {p:.3g}" if p > 0 else f"Mann-Whitney U = {u:,.0f}, p-value < 1e-300 (effectively 0)")
print(f"Rank-biserial effect size = {rbc:.2f}  (|r| ~0.1 small, ~0.3 medium, ~0.5 large)")

Mann-Whitney U = 103,308,952, p-value < 1e-300 (effectively 0)
Rank-biserial effect size = -0.64  (|r| ~0.1 small, ~0.3 medium, ~0.5 large)


In [3]:
rng = np.random.default_rng(42)
B = 2000
diffs = np.array([rng.choice(late, len(late)).mean() - rng.choice(ontime, len(ontime)).mean() for _ in range(B)])
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Mean difference (late - on time) = {late.mean() - ontime.mean():.2f} stars, 95% bootstrap CI [{lo:.2f}, {hi:.2f}]")

Mean difference (late - on time) = -2.02 stars, 95% bootstrap CI [-2.06, -1.98]


In [4]:
# Does the size of the delay matter? Spearman correlation across delivered orders
rho, p2 = stats.spearmanr(d["delay_days"], d["review_score"])
print(f"Spearman rho (delay days vs review score) = {rho:.2f}, p = {p2:.3g}" if p2 > 0 else f"Spearman rho (delay days vs review score) = {rho:.2f}, p < 1e-300")

d["delay_bucket"] = pd.cut(d["delay_days"], [-1e9, -7, 0, 3, 7, 1e9],
                           labels=["7+ days early", "0-7 early", "1-3 late", "4-7 late", "8+ late"])
print(d.groupby("delay_bucket", observed=True).review_score.agg(["count", "mean"]).round(2))

Spearman rho (delay days vs review score) = -0.18, p < 1e-300
               count  mean
delay_bucket              
7+ days early  75744  4.31
0-7 early      13699  4.17
1-3 late        1852  3.29
4-7 late        1748  2.11
8+ late         2781  1.70


## Conclusion (edit with your numbers)
- Late deliveries have a **statistically significant, large** drop in review score (see U, p, effect size and CI above).
- The longer the delay, the lower the score (Spearman, bucket table).
- **Caveat:** this is an *association*. Late orders may also differ in product type, seller or region. It does not prove lateness *caused* the low scores - say "associated with", not "causes".
- **Business recommendation:** improve delivery-date estimates and fix the slowest states/sellers first; the review gain is the prize.